# CharXiv reasoning questions with LM Studio

This notebook runs the same reproducible pipeline as `charxiv_pipeline.py`: sample reasoning questions from the CharXiv validation split with a fixed seed, send each chart and question to a local vision model in LM Studio, and save the reference answer, returned reasoning, and model answer to JSON. Set `N` below to choose the sample size.

In [1]:
import sys
from pathlib import Path

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (WORKING_DIR, *WORKING_DIR.parents)
     if (path / 'Experiments' / 'local' / 'charxiv_pipeline.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find Experiments/local/charxiv_pipeline.py from the notebook working directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'Experiments' / 'local'))
from charxiv_pipeline import run_pipeline

# Same seed means the same 25 examples, as long as the dataset version/order is unchanged.
SEED = 42
N = 5
MODEL = 'google/gemma-4-e4b'  # Change this to the model identifier shown by your LM Studio server.
API_URL = 'http://localhost:1234/api/v1/chat'
OUTPUT = PROJECT_ROOT / 'outputs' / 'charxiv_lm_studio_answers.json'

In [2]:
output_path = run_pipeline(
    output_path=OUTPUT,
    n=N,
    seed=SEED,
    model=MODEL,
    api_url=API_URL,
)
print(f'Results saved to {output_path}')

README.md: 0.00B [00:00, ?B/s]

val.parquet:   0%|          | 0.00/66.8M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/91.7M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1323 [00:00<?, ? examples/s]

[1/5] completed: images/1524.jpg
[2/5] completed: images/270.jpg
[3/5] completed: images/56.jpg
[4/5] completed: images/1773.jpg
[5/5] completed: images/640.jpg
Results saved to /Users/piotrjurczyk/Documents/DS/NLP/Hallucinations-in-AI-Explanations/outputs/charxiv_lm_studio_answers.json


In [3]:
import json

results = json.loads(output_path.read_text(encoding='utf-8'))
results['results'][0]  # Inspect the first saved question, model answer, reasoning, and reference answer.

{'sample_number': 1,
 'dataset_index': 654,
 'example_id': 'images/1524.jpg',
 'question': 'Which year does the log hazard rate exhibit the steepest slope for women from age 60 to 100?',
 'correct_answer': '1905',
 'model_answer': 'Based on the chart provided, the log hazard rate for women from age 60 to 100 exhibits an almost perfectly linear and consistently steep slope for both **1885** and **1905**. The slopes for both years are virtually identical across this entire age range.',
 'model_reasoning': '1.  **Identify the goal:** Find the year when the log hazard rate exhibits the steepest slope for women aged 60 to 100.\n\n2.  **Analyze the chart structure:** The image contains four panels.\n    *   Top Left: Women (Age 0-100, line graph)\n    *   Top Right: Men (Age 0-100, line graph)\n    *   Bottom Left: Women (Age 60-100, scatter/line plot) - *This is the relevant panel.*\n    *   Bottom Right: Men (Age 60-100, scatter/line plot)\n\n3.  **Focus on the target panel:** Bottom Left 